In [0]:
%pip install python-dotenv azure-storage-file-datalake azure-identity --quiet
dbutils.library.restartPython()

import os
import uuid
from dotenv import load_dotenv, find_dotenv
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
)


# CONFIGURAÇÃO

load_dotenv(find_dotenv(), override=True)

CLIENT_ID = os.getenv("ADLS_CLIENT_ID")
TENANT_ID = os.getenv("ADLS_TENANT_ID")
CLIENT_SECRET = os.getenv("ADLS_CLIENT_SECRET")
ACCOUNT = os.getenv("STORAGE_ACCOUNT_NAME")
RAW_CONTAINER = os.getenv("ADLS_CONTAINER", "raw")
BRONZE_CONTAINER = os.getenv("ADLS_BRONZE_CONTAINER", RAW_CONTAINER)

if not all([CLIENT_ID, TENANT_ID, CLIENT_SECRET, ACCOUNT]):
    raise ValueError("Existem variáveis pendentes no arquivo .env!")

host = f"{ACCOUNT}.dfs.core.windows.net"

spark_auth = {
    f"fs.azure.account.auth.type.{host}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{host}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{host}": CLIENT_ID,
    f"fs.azure.account.oauth2.client.secret.{host}": CLIENT_SECRET,
    f"fs.azure.account.oauth2.client.endpoint.{host}": f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token",
}

TABELA = {
    "nome": "physical_vendas_caixa",
    "padrao_arquivo": "physical_vendas_caixa",
    "pasta_raw": "batch-data",
    "chave": "id_transacao",
    "coluna_data": "dt_venda",
    "bronze": f"abfss://{BRONZE_CONTAINER}@{host}/bronze/physical_vendas_caixa",
    "log": f"abfss://{BRONZE_CONTAINER}@{host}/metadata/ingestion_runs/physical_vendas_caixa",
    "schema": StructType([
        StructField("id_transacao", StringType(), True),
        StructField("id_loja", IntegerType(), True),
        StructField("id_caixa", IntegerType(), True),
        StructField("id_operador", IntegerType(), True),
        StructField("dt_venda", TimestampType(), True),
        StructField("valor_total_venda", DoubleType(), True),
        StructField("cpf_cliente", StringType(), True),
        StructField("tipo_pagamento", StringType(), True),
    ]),
}


# 2. FUNÇÕES

def listar_arquivos_raw(padrao: str, pasta: str) -> list:
    """Lista os arquivos da RAW diretamente pelo SDK Azure."""
    credencial = ClientSecretCredential(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
    servico = DataLakeServiceClient(f"https://{host}", credential=credencial)
    fs = servico.get_file_system_client(RAW_CONTAINER)

    return [
        f"abfss://{RAW_CONTAINER}@{host}/{p.name}"
        for p in fs.get_paths(path=pasta)
        if not p.is_directory and padrao in p.name
    ]


def ler_raw(caminhos: list, schema):
    return (
        spark.read
        .options(**spark_auth)
        .option("header", "true")
        .schema(schema)
        .csv(caminhos)
        # captura o arquivo de origem agora, antes de dropDuplicates/join
        .withColumn("bronze_source_file", F.col("_metadata.file_path"))
    )


def bronze_existe(caminho: str) -> bool:
    try:
        spark.read.format("delta").options(**spark_auth).load(caminho).limit(1).count()
        return True
    except Exception:
        return False


def manter_apenas_novos(df_raw, caminho_bronze: str, chave: str):
    """Remove duplicados do lote e descarta chaves que já existem."""
    df_raw = df_raw.dropDuplicates([chave])

    if not bronze_existe(caminho_bronze):
        return df_raw

    chaves_existentes = (
        spark.read.format("delta").options(**spark_auth)
        .load(caminho_bronze)
        .select(chave)
        .distinct()
    )
    return df_raw.join(chaves_existentes, on=chave, how="left_anti")


def adicionar_auditoria(df, coluna_data: str):
    return (
        df
        .withColumn("bronze_ingested_at", F.current_timestamp())
        .withColumn("ano", F.year(F.col(coluna_data)))
        .withColumn("mes", F.month(F.col(coluna_data)))
    )


def registrar_execucao(run_id, tabela, qtd_arquivos, lidas, inseridas, caminho_log):
    log = spark.createDataFrame(
        [(run_id, tabela, qtd_arquivos, lidas, inseridas, "SUCCESS")],
        "run_id string, table_name string, files_read int, rows_read long, rows_inserted long, status string",
    ).withColumn("executed_at", F.current_timestamp())

    log.write.format("delta").mode("append").options(**spark_auth).save(caminho_log)
    return log


# 3. PIPELINE

def executar_bronze(cfg: dict):
    run_id = str(uuid.uuid4())

    arquivos = listar_arquivos_raw(cfg["padrao_arquivo"], cfg["pasta_raw"])
    if not arquivos:
        print(f"[SEM ARQUIVOS] Nenhum arquivo encontrado para {cfg['nome']}.")
        return None, None

    df_raw = ler_raw(arquivos, cfg["schema"])
    lidas = df_raw.count()

    df_novos = manter_apenas_novos(df_raw, cfg["bronze"], cfg["chave"])
    df_novos = adicionar_auditoria(df_novos, cfg["coluna_data"])
    inseridas = df_novos.count()

    if inseridas == 0:
        print(f"[SEM ALTERAÇÃO] {cfg['nome']}: {lidas} linhas lidas, nenhuma nova.")
        registrar_execucao(run_id, cfg["nome"], len(arquivos), lidas, 0, cfg["log"])
        return None, None

    (
        df_novos.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .partitionBy("ano", "mes")
        .options(**spark_auth)
        .save(cfg["bronze"])
    )

    df_log = registrar_execucao(run_id, cfg["nome"], len(arquivos), lidas, inseridas, cfg["log"])
    print(f"[INGESTÃO CONCLUÍDA] {cfg['nome']} | lidas: {lidas} | inseridas: {inseridas}")
    return df_novos, df_log


# 4. EXECUÇÃO

df_bronze, df_log = executar_bronze(TABELA)

if df_bronze is not None:
    display(df_bronze.limit(100))
    display(df_log)


